# 📊 01g — Pipeline indicateurs de contexte (DREES ISD)

Construit `dim_indicateurs_contexte.parquet` (dept × année). Lancer
`00_config_commun.ipynb` avant. Code repris et adapté du notebook d'une
collègue (`donnees_geographiques.ipynb`).

Source : DREES, "ISD — Indicateurs de contexte" (indicateurs sociaux
départementaux, socle commun national/départemental sur l'action sociale).
https://www.data.gouv.fr/datasets/isd-indicateurs-de-contexte
Fichier : `data/raw/indicateurs_contexte/601_indicateurs_de_contexte.csv`.

Après étude des métadonnées, les 10 indicateurs sont conservés (sur les ~200 disponibles) :

| Colonne | Indicateur |
|---|---|
| `cont_depenses_brutes_aide_sociale_hab` | Dépenses totales brutes d'aide sociale par habitant `（Absente ?）`|
| `cont_ind_vieillisement_pop` | Indice de vieillissement de la population |
| `cont_part_pop_pole_urbain` | Part de la population vivant dans un pôle urbain |
| `cont_tx_act` | Taux d'activité |
| `cont_part_csp_cadres` | Part des cadres, professions intellectuelles supérieures |
| `pop_m25` | Population < 25 ans |
| `pop_25_64` | Population 25-64 ans |
| `pop_65p` | Population ≥ 65 ans |
| `chom_defm_abc` | Nombre de demandeurs d'emploi catégories A, B, C `(Absente ?)`|
| `chom_pop_age_trav` | Taux de chômage rapporté à la population en âge de travailler |

In [1]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

Config chargée depuis src/config.py : 96 départements | 2020–2025
RAW_DIR    = /Users/siranh/Documents/Data Scientest/projet_liora/data/raw
TABLES_DIR = /Users/siranh/Documents/Data Scientest/projet_liora/data/processed


In [2]:
INDICATEURS_CONSERVES = [
    "cont_depenses_brutes_aide_sociale_hab", "cont_ind_vieillisement_pop", "cont_part_pop_pole_urbain", "cont_tx_act", "cont_part_csp_cadres", "pop_m25", "pop_25_64", "pop_65p", "om_defm_abc", "chom_pop_age_trav"
]


def build_dim_indicateurs_contexte() -> pd.DataFrame:
    """
    Construit dim_indicateurs_contexte à partir du fichier DREES ISD.

    CLÉ PRIMAIRE : dept × annee
    COLONNES : une par indicateur de INDICATEURS_CONSERVES
    """
    fpath = RAW_DIR / "indicateurs_contexte" / "601_indicateurs_de_contexte.csv"
    if not fpath.exists():
        print(f"⚠️  Fichier manquant : {fpath}")
        return pd.DataFrame()

    df = pd.read_csv(fpath, sep=";", encoding="utf-8", low_memory=False,
                     usecols=["Année", "Département", "Dep", "Libellé Indicateur", "id_indicateur", "Valeur"])
    print(f"  Brut : {len(df):,} lignes")

    df = df[df["id_indicateur"].isin(INDICATEURS_CONSERVES)].copy()
    print(f"  Après filtre indicateurs retenus : {len(df):,} lignes")

    df["dept"] = df["Dep"].astype(str).str.strip().str.upper().str.zfill(2)
    df = df[df["dept"].isin(DEPTS)]
    df["Année"] = pd.to_numeric(df["Année"], errors="coerce")
    df = df[(df["Année"] >= ANNEE_DEBUT) & (df["Année"] <= ANNEE_FIN)]
    df["Valeur"] = pd.to_numeric(df["Valeur"], errors="coerce")

    df_wide = df.pivot_table(
        index=["dept", "Année"], columns="id_indicateur", values="Valeur"
    ).reset_index().rename(columns={"Année": "annee"})
    df_wide.columns.name = None
    df_wide["annee"] = df_wide["annee"].astype(int)

    print(f"  ✅ dim_indicateurs_contexte : {df_wide.shape[0]:,} lignes × {df_wide.shape[1]} colonnes")
    print(f"  Départements : {df_wide['dept'].nunique()} | Années : {sorted(df_wide['annee'].unique())}")
    return df_wide


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
dim_indicateurs_contexte = build_dim_indicateurs_contexte()

if not dim_indicateurs_contexte.empty:
    valider_dim_table(dim_indicateurs_contexte, "dim_indicateurs_contexte", cle=["dept", "annee"])
    dim_indicateurs_contexte.to_parquet(TABLES_DIR / "dim_indicateurs_contexte.parquet", index=False)
    print(f"\n✅ Sauvegardé → data/processed/dim_indicateurs_contexte.parquet")
    display(dim_indicateurs_contexte.head(10))

  Brut : 145,933 lignes
  Après filtre indicateurs retenus : 17,663 lignes
  ✅ dim_indicateurs_contexte : 576 lignes × 10 colonnes
  Départements : 96 | Années : [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
── Validation de dim_indicateurs_contexte ──
  ✅ Tous les codes dept sont valides (96 départements)
  ✅ Aucun doublon sur la clé ['dept', 'annee']
  Taux de valeurs manquantes :
    cont_part_csp_cadres            83.3%
    cont_part_pop_pole_urbain       83.3%
    cont_tx_act                     83.3%
  ✅ OK — prêt pour la fusion (dim_indicateurs_contexte)


✅ Sauvegardé → data/processed/dim_indicateurs_contexte.parquet


,dept,annee,chom_pop_age_trav,cont_ind_vieillisement_pop,cont_part_csp_cadres,cont_part_pop_pole_urbain,cont_tx_act,pop_25_64,pop_65p,pop_m25
0,01,2020,411070.0,70.31,NaN,NaN,NaN,337749.0,119679.0,200428.0
1,01,2021,415353.0,72.00,NaN,NaN,NaN,339802.0,122369.0,201031.0
2,01,2022,419688.0,72.99,0.100,0.2023,0.790,343132.0,124877.0,203280.0
3,01,2023,423321.0,75.06,NaN,NaN,NaN,347276.0,128515.0,203553.0
4,01,2024,426826.0,76.52,NaN,NaN,NaN,350238.0,131582.0,204047.0
5,01,2025,431008.0,78.18,NaN,NaN,NaN,353108.0,134758.0,204300.0
6,02,2020,317636.0,85.06,NaN,NaN,NaN,259957.0,111645.0,157772.0
7,02,2021,315874.0,86.48,NaN,NaN,NaN,258545.0,112435.0,156488.0
8,02,2022,313792.0,88.66,0.048,0.3479,0.734,256609.0,113686.0,155263.0
9,02,2023,311727.0,90.79,NaN,NaN,NaN,255068.0,114773.0,153501.0
